# Dephasing Matrix Test

## What is tested

`get_transformed_dephasing_matrix` returns the coherence decay rates of a context. This
notebook checks that the returned matrix equals the coherence diagonal of the relaxation
superoperator (thermal plus driven) and that it obeys the physical constraints: zero
diagonal, symmetry and nonnegative rates. The checks are done for single contexts and for
contexts combined by concatenation and multiplication.

Every test uses four kinds of relaxation (thermal rates only, explicit dephasing, decay
without population transfer, and thermal plus driven rates) and two combinations of bases.

## List of tests

**Simple tests** (`run_tests_simple`)

1. `test_dephasing_single_context`: the dephasing matrix of single contexts.

**Difficult tests** (`run_tests_difficult`)

2. `test_dephasing_concatenation`: the dephasing matrix of contexts combined with `mars.concat`.
3. `test_dephasing_multiplication`: the dephasing matrix of contexts combined with `@`.

## How to run

Run all cells. The last two cells call `run_tests_simple()` and `run_tests_difficult()`.
Each test prints one line when it passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys
import typing as tp

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
import mars.population.transform as transform
from mars import spin_model
from mars.population.contexts import Context

dtype = torch.float64
FIELD = 5.1  # T

## Helper functions

### Samples and bases

In [2]:
def create_sample(g_values: tuple, zfs_values: tuple) -> spin_model.SolidSample:
    """Create a powder sample of an S = 1 system with the given g-tensor and (D, E) in Hz."""
    system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[spin_model.Interaction(g_values, dtype=dtype)],
        electron_electron=[(0, 0, spin_model.DEInteraction(zfs_values, dtype=dtype))],
        dtype=dtype,
    )
    return spin_model.SolidSample(base_spin_system=system, gauss=0.001, lorentz=0.001, mesh=(4, 4), dtype=dtype)


def create_samples() -> tuple[spin_model.SolidSample, spin_model.SolidSample]:
    """Create the two different samples used by all tests."""
    sample_1 = create_sample((2.02, 2.04, 2.06), (200e6, 50e6))
    sample_2 = create_sample((2.02, 2.14, 2.16), (200e6, 70e6))
    return sample_1, sample_2


def get_eigen_basis(sample: spin_model.SolidSample, field: float) -> torch.Tensor:
    """Return the eigenvectors of the sample Hamiltonian at the given field."""
    F, _, _, Gz = sample.get_hamiltonian_terms()
    hamiltonian = F + Gz * torch.tensor(field, dtype=dtype)
    _, vectors = torch.linalg.eigh(hamiltonian.unsqueeze(-3))
    return vectors


def direct_sum_batched(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Return the block-diagonal direct sum of two batches of square matrices."""
    n, m = a.shape[-1], b.shape[-1]
    result = torch.zeros(*a.shape[:-2], n + m, n + m, dtype=a.dtype, device=a.device)
    result[..., :n, :n] = a
    result[..., n:, n:] = b
    return result

### Contexts

In [3]:
def rates(values: list[list[float]]) -> torch.Tensor:
    """Convert a nested list of rates to a tensor."""
    return torch.tensor(values, dtype=dtype)


def contexts_thermal_only(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with thermal rates only."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 200.0], [0.0, 200.0, 0.0]]),
        dtype=dtype,
    )
    return context_1, context_2


def contexts_explicit_dephasing(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with decay, thermal rates and explicit dephasing."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.25, 0.25],
        decay_rates=[1000.0, 200.3, 100.2],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        dephasing=[1e4, 1e5, 1e6],
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[111.0, 1234.0, 1232.0],
        thermal_rates=rates([[0.0, 100.0, 0.0], [100.0, 0.0, 200.0], [0.0, 200.0, 0.0]]),
        dephasing=[2e5, 1e4, 1e3],
        dtype=dtype,
    )
    return context_1, context_2


def contexts_decay_only(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with decay and zero thermal rates."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 200.3, 100.2],
        thermal_rates=torch.zeros(3, 3, dtype=dtype),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[111.0, 1234.0, 1232.0],
        thermal_rates=torch.zeros(3, 3, dtype=dtype),
        dtype=dtype,
    )
    return context_1, context_2


def contexts_thermal_and_driven(sample_1, sample_2, basis_1, basis_2):
    """Create two contexts with thermal and driven rates."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        decay_rates=[0.0, 200.3, 100.2],
        thermal_rates=rates([[0.0, 50.0, 0.0], [50.0, 0.0, 50.0], [0.0, 50.0, 0.0]]),
        driven_rates=rates([[0.0, 25.0, 0.0], [25.0, 0.0, 25.0], [0.0, 25.0, 0.0]]),
        dtype=dtype,
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3],
        decay_rates=[0.0, 0.0, 0.0],
        thermal_rates=rates([[0.0, 50.0, 0.0], [50.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        driven_rates=rates([[0.0, 25.0, 0.0], [25.0, 0.0, 50.0], [0.0, 0.0, 0.0]]),
        dtype=dtype,
    )
    return context_1, context_2


CONTEXT_CREATORS = [
    contexts_thermal_only,
    contexts_explicit_dephasing,
    contexts_decay_only,
    contexts_thermal_and_driven,
]


def context_pairs(sample_1, sample_2) -> tp.Iterator[tuple[Context, Context]]:
    """Yield every kind of context pair in two combinations of bases."""
    for creator in CONTEXT_CREATORS:
        yield creator(sample_1, sample_2, "xyz", "zeeman")
        yield creator(sample_1, sample_2, "zfs", "multiplet")

### Common check

In [4]:
def check_dephasing_matrix(
    context: Context, vectors: torch.Tensor, atol: float = 1e-6, rtol: float = 1e-5
) -> None:
    """Compare the dephasing matrix with the superoperator diagonal and check its physical constraints."""
    thermal = context.get_transformed_thermal_superop(vectors)
    driven = context.get_transformed_driven_superop(vectors)
    dephasing = context.get_transformed_dephasing_matrix(vectors)

    # Without any relaxation source the method must return None
    if thermal is None and driven is None and context.dephasing is None:
        assert dephasing is None, "Expected None when no dephasing sources are defined"
        return

    superoperators = [superop for superop in (thermal, driven) if superop is not None]
    expected = transform.extract_dephasing_matrix_from_superoperator(sum(superoperators))

    diagonal = torch.diagonal(dephasing, dim1=-2, dim2=-1)
    assert torch.allclose(diagonal, torch.zeros_like(diagonal), atol=atol), "The diagonal must be zero"
    assert torch.allclose(dephasing, dephasing.transpose(-2, -1), atol=atol, rtol=rtol), (
        "The dephasing matrix must be symmetric"
    )
    assert torch.all(dephasing.real >= -1e-8), f"Negative dephasing rate: {dephasing.real.min().item():.2e}"
    assert torch.allclose(dephasing, expected, atol=atol, rtol=rtol), (
        f"The dephasing matrix differs from the superoperator diagonal. "
        f"Max difference: {(dephasing - expected).abs().max().item():.2e}"
    )

## Tests

In [5]:
def test_dephasing_single_context():
    """Check the dephasing matrix of single contexts."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, FIELD)
    vectors_2 = get_eigen_basis(sample_2, FIELD)

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_dephasing_matrix(context_1, vectors_1)
        check_dephasing_matrix(context_2, vectors_2)
    print("Single context test passed (the matrix matches the superoperator and is physical)")


def test_dephasing_concatenation():
    """Check the dephasing matrix of two concatenated contexts."""
    sample_1, sample_2 = create_samples()
    vectors = direct_sum_batched(get_eigen_basis(sample_1, FIELD), get_eigen_basis(sample_2, FIELD))

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_dephasing_matrix(mars.concat([context_1, context_2]), vectors)
    print("Concatenation test passed (the matrix matches the superoperator and is physical)")


def test_dephasing_multiplication():
    """Check the dephasing matrix of the product of two contexts."""
    sample_1, sample_2 = create_samples()
    vectors = transform.batched_kron(get_eigen_basis(sample_1, FIELD), get_eigen_basis(sample_2, FIELD))

    for context_1, context_2 in context_pairs(sample_1, sample_2):
        check_dephasing_matrix(context_1 @ context_2, vectors)
    print("Multiplication test passed (the matrix matches the superoperator and is physical)")

## Run the tests

In [6]:
def run_tests_simple():
    """Run the tests for single contexts."""
    test_dephasing_single_context()
    print("\nThe simple dephasing matrix test passed.")


def run_tests_difficult():
    """Run the tests for combined contexts."""
    tests = [test_dephasing_concatenation, test_dephasing_multiplication]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} difficult dephasing matrix tests passed.")

In [7]:
run_tests_simple()

Single context test passed (the matrix matches the superoperator and is physical)

The simple dephasing matrix test passed.


In [8]:
run_tests_difficult()

D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is not described. This is valid only in the slow-exchange limit, when the transition rates between subsystems are much smaller than the differences between their resonance frequencies.
  warnings.warn(


Concatenation test passed (the matrix matches the superoperator and is physical)
Multiplication test passed (the matrix matches the superoperator and is physical)

All 2 difficult dephasing matrix tests passed.


D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is not described. This is valid only in the slow-exchange limit, when the transition rates between subsystems are much smaller than the differences between their resonance frequencies.
  warnings.warn(
